# BUSINESS UNDERSTANDING.
International football tournaments generate billions in global engagement yet most publicly available prediction models rely solely on historical win-loss records. The aim of this project is to develop and evaluate a multi-feature probabilistic prediction system for the FIFA World Cup 2026 knockout stage by integrating historical match performance, Elo ratings, Poisson goal-scoring models, squad market values, and managerial factors to generate pre-match win probabilities and assess their predictive accuracy against actual tournament outcomes.

## Problem Statement.
Existing football prediction models frequently rely on historical match results or ranking systems alone, often overlooking important factors such as squad market value, team strength, recent form, and managerial stability. As a result, their ability to accurately predict outcomes in major tournaments such as the FIFA World Cup may be limited. Additionally, many models are evaluated only on historical data rather than through real-time prediction and validation. This project addresses this gap by developing a probabilistic prediction system that integrates Elo ratings, Poisson goal-scoring models, squad market values, and managerial factors to generate and evaluate pre-match predictions for the FIFA World Cup 2026 knockout stage.

## Objectives.
`Main Objective.`
* To design, implement, and validate a data-driven model capable of predicting FIFA World Cup 2026 knockout-stage match outcomes using historical and contemporary football performance indicators.
 
`Specific Objectives`
* To collect and integrate historical international match results, Elo ratings, squad market values, and managerial information relevant to FIFA World Cup teams.
* To preprocess and harmonize football datasets by resolving team name inconsistencies, missing values, and feature mismatches across multiple data sources.
* To assess the contribution of squad market value and managerial stability variables in improving prediction accuracy compared to models based solely on historical performance indicators.
* To evaluate the predictive performance of the proposed model using historical World Cup data and selected validation metrics such as accuracy

# DATA UNDERSTANDING.
* The data used for analysis are:

    **Historical Match Results**: It provided data on all the team has ever played.
    
    **Elo Ratings**: Used to check the strength of the team.
    
    **Market Values**: This captures player quality in the teams.
    
    **Live 2026 fixtures**: This is the real time feed for the actual tornament.
    
    **Team names**: It is used to ensure name standardization in the teams.

## Data Description.

In [1]:
# import all the necessary libraries.
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os

In [2]:
# load results dataset
results = pd.read_csv(r"C:\Users\Administrator\OneDrive\Desktop\My projects\World Cup 2026\results.csv")
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [3]:
# load the ELO rating dataset
elo= pd.read_csv(r"C:\Users\Administrator\OneDrive\Desktop\My projects\World Cup 2026\elo_ratings_wc2026.csv")
elo.head()

,year,snapshot_date,country,rank,country_code,rating,rank_max,rating_max,rank_avg,rating_avg,...,matches_home,matches_away,matches_neutral,wins,losses,draws,goals_for,goals_against,confederation,is_host
0,1901,1901-12-31,England,1,EN,2013,1,2079,2,1989,...,38,35,0,46,16,11,262,102,UEFA,0
1,1901,1901-12-31,Scotland,2,SQ,1973,1,2104,1,2018,...,37,37,0,53,9,12,277,101,UEFA,0
2,1902,1902-12-31,Argentina,1,AR,2021,1,2021,1,2021,...,0,1,0,1,0,0,6,0,CONMEBOL,0
3,1902,1902-12-31,England,2,EN,1995,1,2079,2,1989,...,39,38,0,47,16,14,266,105,UEFA,0
4,1902,1902-12-31,Scotland,3,SQ,1983,1,2104,1,2017,...,39,40,0,56,9,14,293,106,UEFA,0


In [4]:
# load the narket values dataset
market_values= pd.read_csv(r"C:\Users\Administrator\OneDrive\Desktop\My projects\World Cup 2026\market_values.csv")
market_values.head()

,team,total_mv_millions,year
0,France,1520.0,2026
1,England,1360.0,2026
2,Spain,1220.0,2026
3,Portugal,1010.0,2026
4,Germany,947.0,2026


In [5]:
# load the team name dataset
team_names = pd.read_csv(r"C:\Users\Administrator\OneDrive\Desktop\My projects\World Cup 2026\team_names.csv")
team_names.head()

,former_name,canonical_name,notes
0,South Korea,Korea Republic,Common English name → FIFA name
1,North Korea,Korea DPR,Common English name → FIFA name
2,Ivory Coast,Côte d'Ivoire,English → French official FIFA name
3,DR Congo,Congo DR,Word order differs across datasets
4,Zaire,Congo DR,Former name 1971-1997 in Kaggle data


In [6]:
team_names[
    team_names["former_name"].str.contains("Korea", case=False, na=False) |
    team_names["canonical_name"].str.contains("Korea", case=False, na=False)
]

,former_name,canonical_name,notes
0,South Korea,Korea Republic,Common English name → FIFA name
1,North Korea,Korea DPR,Common English name → FIFA name
14,Korea Republic of,Korea Republic,Grammatical variant
15,Republic of Korea,Korea Republic,Grammatical variant
43,Korea Republic,South Korea,Already canonical


In [7]:
# Information summary of the 4 datasets
datasets = {
    "Results": results,
    "Elo Ratings": elo,
    "Market Value": market_values,
    "Teams": team_names
}

report = []

for name, df in datasets.items():
    report.append({
        "Dataset": name,
        "Rows": df.shape[0],       #to chek len of rows
        "Columns": df.shape[1],    # to check len of columns
        "Missing Values": df.isnull().sum().sum(),
        "Duplicates": df.duplicated().sum(),
        "Numeric Columns": len(df.select_dtypes(include='number').columns),     # No. of numerical columns in the datasets.
        "Categorical Columns": len(df.select_dtypes(exclude='number').columns)  # No.of categorical columns in the datasets.
    })

report_df = pd.DataFrame(report)

display(report_df)

,Dataset,Rows,Columns,Missing Values,Duplicates,Numeric Columns,Categorical Columns
0,Results,49477,9,88,0,2,7
1,Elo Ratings,4683,23,0,0,19,4
2,Market Value,48,3,0,0,2,1
3,Teams,53,3,0,0,0,3


# DATA PREPARATION.
In this Phase the datasets will be transformed in to a structuredand clean format that can be used for modeling. This step ensures that the dataset is free from inconsistencies, missing values, and unnecessary variables while preparing it for machine learning and deep learning algorithms.

## Data Cleaning.

### Result dataset
* Handle missing values.
* Drop unnecessary columns.
* Stanadardize team names.
* Convert datse to datelines.


In [8]:
# laod the dataset
results = pd.read_csv(r"C:\Users\Administrator\OneDrive\Desktop\My projects\World Cup 2026\results.csv")
# create a copies of the dataset
df_result = results.copy(deep=True)
elo_copy = elo.copy()
market_copy = market_values.copy()
tn_copy = team_names.copy()

print("Working copies created successfully.")

Working copies created successfully.


In [9]:
#result dataset
df_result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  object 
 1   home_team   49477 non-null  object 
 2   away_team   49477 non-null  object 
 3   home_score  49433 non-null  float64
 4   away_score  49433 non-null  float64
 5   tournament  49477 non-null  object 
 6   city        49477 non-null  object 
 7   country     49477 non-null  object 
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), object(6)
memory usage: 3.1+ MB


* In this dataset I have cleaned it by:
   
       Changing the date data type from object to to datetimes.
   
       Handling missing values.
       
       Standardize team names.

In [10]:
# change data type of date
df_result['date']= pd.to_datetime(df_result['date'])
print("Dtypes: ")
print(df_result.dtypes)

Dtypes: 
date          datetime64[ns]
home_team             object
away_team             object
home_score           float64
away_score           float64
tournament            object
city                  object
country               object
neutral                 bool
dtype: object


In [11]:
# verify the timeframe that dataset covers.
print("Date range:", df_result['date'].min().date(), "→", df_result['date'].max().date())

Date range: 1872-11-30 → 2026-06-27


In [12]:
# checking for missing values.
df_result.isnull().sum()

date           0
home_team      0
away_team      0
home_score    44
away_score    44
tournament     0
city           0
country        0
neutral        0
dtype: int64

`Missing values were assessed on a variable-by-variable basis. Future fixtures with missing match scores were retained during data preparation because the scores were unavailable at the time of analysis. These records were excluded from model training and reserved for prediction. Missing values resulting from unmatched team names were addressed through team-name standardization rather than statistical imputation.`

In [13]:
# check for duplicates
df_result.duplicated().sum()           

0

In [14]:
# elo rating dataset
elo_copy.info()                 # the dataset has no missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4683 entries, 0 to 4682
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   year             4683 non-null   int64 
 1   snapshot_date    4683 non-null   object
 2   country          4683 non-null   object
 3   rank             4683 non-null   int64 
 4   country_code     4683 non-null   object
 5   rating           4683 non-null   int64 
 6   rank_max         4683 non-null   int64 
 7   rating_max       4683 non-null   int64 
 8   rank_avg         4683 non-null   int64 
 9   rating_avg       4683 non-null   int64 
 10  rank_min         4683 non-null   int64 
 11  rating_min       4683 non-null   int64 
 12  matches_total    4683 non-null   int64 
 13  matches_home     4683 non-null   int64 
 14  matches_away     4683 non-null   int64 
 15  matches_neutral  4683 non-null   int64 
 16  wins             4683 non-null   int64 
 17  losses           4683 non-null   

In [15]:
# change date to datetime
elo_copy["snapshot_date"]= pd.to_datetime(elo_copy["snapshot_date"])
print("Dtypes: ")
print(elo_copy.dtypes)

Dtypes: 
year                        int64
snapshot_date      datetime64[ns]
country                    object
rank                        int64
country_code               object
rating                      int64
rank_max                    int64
rating_max                  int64
rank_avg                    int64
rating_avg                  int64
rank_min                    int64
rating_min                  int64
matches_total               int64
matches_home                int64
matches_away                int64
matches_neutral             int64
wins                        int64
losses                      int64
draws                       int64
goals_for                   int64
goals_against               int64
confederation              object
is_host                     int64
dtype: object


In [16]:
# verify timeframe that the datset covers.
print("Date range:", elo_copy['snapshot_date'].min().date(), "→", elo_copy['snapshot_date'].max().date())

Date range: 1901-12-31 → 2026-12-31


In [17]:
# check for duplacates
elo_copy.duplicated().sum()

0

In [18]:
# clean the team name datset...
team_names = team_names.drop(index= 43)      # drop South Korea Row to prevent contadicting

In [19]:
# Create output folder (prevents OSError)
os.makedirs("data/processed", exist_ok=True)

# remove spacing
df_result["home_team"] = df_result["home_team"].str.strip()
df_result["away_team"] = df_result["away_team"].str.strip()

elo_copy["country"] = elo_copy["country"].str.strip()

market_copy["team"] = market_copy["team"].str.strip()

tn_copy["former_name"] = tn_copy["former_name"].str.strip()
tn_copy["canonical_name"] = tn_copy["canonical_name"].str.strip()

# Create team mapping dictionary
team_map = dict(
    zip(
        tn_copy["former_name"],
        tn_copy["canonical_name"]
    )
)

# Standardize team names
df_result["home_team"] = df_result["home_team"].replace(team_map)
df_result["away_team"] = df_result["away_team"].replace(team_map)

elo_copy["country"] = elo_copy["country"].replace(team_map)

market_copy["team"] = market_copy["team"].replace(team_map)

# Check unmatched teams
results_teams = set(df_result["home_team"]).union(
    set(results["away_team"])
)

elo_teams = set(elo_copy["country"])

market_teams = set(market_copy["team"])

print("=" * 60)
print("Teams in Results but NOT in Elo:")
print(sorted(results_teams - elo_teams))

print("\n" + "=" * 60)
print("Teams in Elo but NOT in Results:")
print(sorted(elo_teams - results_teams))

print("\n" + "=" * 60)
print("Teams in Results but NOT in Market Values:")
print(sorted(results_teams - market_teams))

print("\n" + "=" * 60)
print("Teams in Market Values but NOT in Results:")
print(sorted(market_teams - results_teams))

# Save cleaned datasets
df_result.to_csv("data/processed/results_standardized.csv", index=False)
elo_copy.to_csv("data/processed/elo_standardized.csv", index=False)
market_copy.to_csv("data/processed/market_values_standardized.csv", index=False)

print("\nDatasets saved successfully!")

print("\nSaved files:")
print("- data/processed/results_standardized.csv")
print("- data/processed/elo_standardized.csv")
print("- data/processed/market_values_standardized.csv")


Teams in Results but NOT in Elo:
['Abkhazia', 'Afghanistan', 'Albania', 'Alderney', 'Ambazonia', 'American Samoa', 'Andalusia', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Arameans Suryoye', 'Armenia', 'Artsakh', 'Aruba', 'Asturias', 'Aymara', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barawa', 'Barbados', 'Basque Country', 'Belarus', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Biafra', 'Bolivia', 'Bonaire', 'Botswana', 'British Virgin Islands', 'Brittany', 'Brunei', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Canary Islands', 'Cape Verde', 'Cascadia', 'Catalonia', 'Cayman Islands', 'Central African Republic', 'Central Spain', 'Chad', 'Chagos Islands', 'Chameria', 'Chechnya', 'Chile', 'China', 'Cilento', 'Comoros', 'Congo', 'Cook Islands', 'Corsica', 'Costa Rica', 'County of Nice', 'Crimea', 'Cuba', 'Cyprus', 'Czech Republic', 'Czechoslovakia', 'DR Congo', 'Darfur', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Donetsk PR', 'Délvidék', 


Datasets saved successfully!

Saved files:
- data/processed/results_standardized.csv
- data/processed/elo_standardized.csv
- data/processed/market_values_standardized.csv


In [20]:
# seperate future fixtures in result
# Clean the market value dataset
# merge the dataset

['Abkhazia', 'Afghanistan', 'Albania', 'Alderney', 'Algeria', 'Ambazonia', 'American Samoa', 'Andalusia', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Arameans Suryoye', 'Argentina', 'Armenia', 'Artsakh', 'Aruba', 'Asturias', 'Australia', 'Austria', 'Aymara', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barawa', 'Barbados', 'Basque Country', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Biafra', 'Bolivia', 'Bonaire', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'British Virgin Islands', 'Brittany', 'Brunei', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Canary Islands', 'Cape Verde', 'Cascadia', 'Catalonia', 'Cayman Islands', 'Central African Republic', 'Central Spain', 'Chad', 'Chagos Islands', 'Chameria', 'Chechnya', 'Chile', 'China', 'Cilento', 'Colombia', 'Comoros', 'Congo', 'Congo DR', 'Cook Islands', 'Corsica', 'Costa Rica', 'County of Nice', 'Crimea', 'Croatia', 'Cuba', 'Curaçao', 'Cyprus', 'Czech Re